In [4]:
import os
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import plotly.express as px
import statistics
from datetime import datetime, date, timedelta

In [ ]:
#Read the data into a data frame
df = pd.read_csv("extract-3-very-clean.csv", low_memory=False)

/var/folders/m2/2zh5b68d5b92gmyhffzbjxgr0000gn/T/ipykernel_13379/915348292.py:2: DtypeWarning: Columns (0: Property ID, 1: Download date / time, 2: Property name, 3: Property unit number, 4: Settlement date, 5: Nature of property, 6: Primary purpose, 7: Dealing number) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("extract-3-very-clean.csv")


In [ ]:
#See how many records are included
df.size

In [ ]:
#Change date fields to datetime type
df['Contract date']= pd.to_datetime(df['Contract date'])
df['Settlement date']= pd.to_datetime(df['Settlement date'])

#Then check types are okay
df.dtypes

In [ ]:
#Filter the dataset to your own search area
#(could obvs filter by whatever, but this is my search area)

property_locations = ['Lawson','Hazelbrook','Woodford','Linden','Faulconbridge','Springwood','Valley Heights','Warrimoo']
property_streetname = None #e.g.: ['Railway Ave']
exclude_zoning = ['IN1', 'IN2', 'I', 'B', 'B1', 'B2', 'B7']
exclude_primary_purpose = ['Service stations', 'Service stati', 'Service statio', 'Shop', 'Hall', 'Commercial']
include_only_primary_purpose = None #e.g.: ['Vacant land']
postcode_min = 2750
postcode_max = 2800
price_min = None
price_max = 5000000
area_min = 100
area_max = None
start_date = '1990-01-01'
end_date = '2100-01-01'

#Go ahead and implement all of the above filters
df_myarea = df
if property_locations: df_myarea = df_myarea[ df_myarea['Property locality'].isin(property_locations) ] #In location specified
if property_streetname: df_myarea = df_myarea[ df_myarea['Property street name'].isin(property_streetname) ] #In street name exactly specified
if area_min: df_myarea = df_myarea[ df_myarea['Area'] >= area_min ] #More than minimum area size
if area_max: df_myarea = df_myarea[ df_myarea['Area'] <= area_max ] #Less than maximum area size
if postcode_min: df_myarea = df_myarea[ df_myarea['Property post code'] >= postcode_min ] #In postcode range
if postcode_max: df_myarea = df_myarea[ df_myarea['Property post code'] <= postcode_max ] #In postcode range
if price_min: df_myarea = df_myarea[ df_myarea['Purchase price'] >= price_min ] #More than minimum price
if price_max: df_myarea = df_myarea[ df_myarea['Purchase price'] <= price_max ] #Less than maximum price
if exclude_zoning: df_myarea = df_myarea[ ~df_myarea['Zoning'].isin(exclude_zoning) ] #Exclude weird zoning types
if start_date: df_myarea = df_myarea[ df_myarea['Contract date'] >= start_date] #Make sure all values are in the correct date range
if end_date: df_myarea = df_myarea[ df_myarea['Contract date'] <= end_date] #Make sure all values are in the correct date range
if exclude_primary_purpose: df_myarea = df_myarea[ ~df_myarea['Primary purpose'].isin(exclude_primary_purpose) ] #Exclude weird zoning types
if include_only_primary_purpose: df_myarea = df_myarea[ df_myarea['Primary purpose'].isin(include_only_primary_purpose) ] #Include only these zoning types

print(str(len(df_myarea.index)) + ' records kept')

In [ ]:
#Show zoning and purpose types in the dataset
#Types: https://www.valuergeneral.nsw.gov.au/__data/assets/pdf_file/0019/216406/Property_Sales_Data_File_Zone_Codes_and_Descriptions_V2.pdf

display(df_myarea['Primary purpose'].unique())
display(df_myarea['Zoning'].unique())

In [ ]:
#Fix NaNs
df_myarea['Zoning'].fillna(value='None', inplace=True)
df_myarea['Area'].fillna(value=0, inplace=True)
df_myarea['Purchase price'].fillna(value=0, inplace=True)

In [ ]:
#Remove purchase price outliers

before=len(df_myarea.index)

#Display the outliers
display(df_myarea[(np.abs(stats.zscore(df_myarea['Purchase price'])) >= 3)])

#Remove them from the data
#df_myarea = df_myarea[(np.abs(stats.zscore(df_myarea['Purchase price'])) < 3)]

after=len(df_myarea.index)
print('Removed ' + str(before-after) + ' outliers (more than 3 standard deviations from the mean).')

In [ ]:
#Price histogram in ~$50K bins

if not df_myarea.empty:
    fig = px.histogram(df_myarea, x="Purchase price", nbins=int(df_myarea['Purchase price'].max()/50000),
        title='Price histogram', width=1000, height=400,
    )
    fig.show()


In [ ]:
#Show the top 5 most expensive properties in the area

df_myarea.sort_values('Purchase price', ascending=False).head(5)

In [ ]:
#Show all properties with a purchase price of 0 (these are likely missing values)

df_myarea[df_myarea['Purchase price'] == 0]

In [ ]:
#Display all the records.
display(df_myarea)

In [ ]:
#Price by size and contract date

#Scale property size so the dots don't get too small
median = statistics.median(df_myarea['Area'])
df_myarea['Area - scaled'] = [(x - median) / 1000 + median for x in df_myarea['Area']]

fig = px.scatter(
    df_myarea,
    x='Contract date',
    y='Purchase price',    
    size='Area - scaled',
    color='Zoning',
    title='Price and size of property by contract date',
    width=1000,
    height=500,
    labels={'x':'Contract date'},
    hover_name=df_myarea['Property house number'] + ' ' + df_myarea['Property street name'] + ', ' + df_myarea['Property locality'],
    hover_data={
        'Area - scaled':False,
        'Zoning':True,
        'Area':True
    }
)

fig.show()

In [ ]:
#Price by contract date

fig = px.scatter(
    df_myarea,
    x='Contract date',
    y='Purchase price',    
    title='Price over time',
    trendline='rolling',
    trendline_options=dict(window=45),    
    trendline_color_override="red",
    width=1000,
    height=500,
    labels={'x':'Contract date'},
    hover_name=df_myarea['Property house number'] + ' ' + df_myarea['Property street name'] + ', ' + df_myarea['Property locality'],
    hover_data={
        'Area - scaled':False,
        'Zoning':True,
        'Area':True
    }
)

fig.show()

In [ ]:
#Median price by contract date

df_myarea_agg=df_myarea[['Contract date','Purchase price']]
df_myarea_agg=df_myarea_agg.groupby(['Contract date']).median()
#This is the same as above:
##df_myarea_agg=df_myarea_agg.groupby(([pd.Grouper(key='Contract date', freq='D')])).median()

fig = px.scatter(
    df_myarea_agg,
    x=df_myarea_agg.index.values,
    y='Purchase price',    
    title='Median price by date',
    width=1000,
    height=500,
    labels={'x':'Contract date'},
)

fig.show()

In [ ]:
#Get last downloaded file date and set variable 90 days before that
d = datetime.now()
offset = -datetime.now().weekday() - 7 - 90
this_date = (datetime.now() + timedelta(offset))
print(this_date)

In [ ]:
#Monthly median price

df_myarea_aggM = df_myarea[['Contract date', 'Purchase price']]
df_myarea_aggM = df_myarea_aggM.groupby([pd.Grouper(key='Contract date', freq='ME')]).agg('median')

df_myarea_aggM['Rolling 6-month median'] = df_myarea_aggM.rolling(6).median()

#Could also do this if we wanted to show multiple types - e.g. mean, sum, etc
#g1 = df_myarea_m.groupby(pd.Grouper(key='Contract date', freq="M")).median()
#g2 = df_myarea_m.groupby(pd.Grouper(key='Contract date', freq="M")).mean()
#g = g1.merge(g2, left_on='Contract date', right_on='Contract date', suffixes=(' median', ' mean'))

fig = px.line(
    df_myarea_aggM,
    title='Monthly median purchase price',
    width=1000,
    height=500
)

fig.add_vline(x=this_date, line_width=2, line_dash="dot", line_color="green")
fig.show()

In [ ]:
#Sales volume by month

latest_date = df_myarea['Contract date'].max() - timedelta(days=90)

df_myarea_aggMc = df_myarea[['Contract date', 'Purchase price']]
df_myarea_aggMc = df_myarea_aggMc.groupby([pd.Grouper(key='Contract date', freq='M')]).agg('count')
df_myarea_aggMc.rename(columns={'Purchase price':'Number of sales'}, inplace=True)
df_myarea_aggMc['Rolling 6-month median'] = df_myarea_aggMc.rolling(6).median()


fig = px.line(
    df_myarea_aggMc,
    title='Sales volume by month',
    width=1000,
    height=500
)

fig.add_vline(x=this_date, line_width=2, line_dash="dot", line_color="green")
fig.show()

---
## Street-Level Query with Spatial Map

Query all sales on a specific street within a date range and plot each sale on an interactive map.

Addresses are geocoded via OpenStreetMap (Nominatim) and cached locally in `geocode_cache.csv` so subsequent runs are instant.

> **Requirement:** `geopy` — already in `requirements.txt`.

In [ ]:
# --- Parameters: edit these to query any street and date range ---

query_street     = 'King St'       # title case; partial match (catches 'King St', 'King Street', etc.)
query_locality   = 'Mascot'        # suburb name, title case
query_postcode   = 2020            # set to None to skip
query_start_date = '2020-01-01'
query_end_date   = '2020-12-31'

In [ ]:
# Filter the main dataframe to the street query
df_street = df[
    df['Property street name'].str.contains(query_street, case=False, na=False) &
    (df['Property locality'].str.upper() == query_locality.upper()) &
    (df['Contract date'] >= query_start_date) &
    (df['Contract date'] <= query_end_date)
].copy()

if query_postcode is not None:
    df_street = df_street[df_street['Property post code'] == float(query_postcode)]

df_street = df_street.sort_values('Contract date').reset_index(drop=True)

print(f"{len(df_street)} sales found on {query_street}, {query_locality} "
      f"between {query_start_date} and {query_end_date}")
df_street[['Property house number','Property street name','Property locality',
           'Contract date','Purchase price','Area','Zoning']]

In [ ]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

GEOCODE_CACHE = 'geocode_cache.csv'

# Load cache so we never hit the API twice for the same address
_cache = (
    pd.read_csv(GEOCODE_CACHE, index_col='address')
    if os.path.exists(GEOCODE_CACHE)
    else pd.DataFrame(columns=['address', 'lat', 'lng']).set_index('address')
)

_geolocator = Nominatim(user_agent='nsw-property-sales-analysis')
_geocode    = RateLimiter(_geolocator.geocode, min_delay_seconds=1, error_wait_seconds=5)

def _geocode_row(row):
    """Return (lat, lng) for a property row, reading from cache first."""
    postcode = int(row['Property post code']) if pd.notna(row['Property post code']) else ''
    address  = f"{row['Property house number']} {row['Property street name']}, "\
               f"{row['Property locality']} NSW {postcode}, Australia"
    if address in _cache.index:
        return _cache.loc[address, 'lat'], _cache.loc[address, 'lng']
    try:
        loc = _geocode(address)
        lat, lng = (loc.latitude, loc.longitude) if loc else (None, None)
    except Exception as e:
        print(f'Geocoding failed for {address}: {e}')
        lat, lng = None, None
    _cache.loc[address] = [lat, lng]
    _cache.to_csv(GEOCODE_CACHE)
    return lat, lng

print(f'Geocoding {len(df_street)} addresses (cached results are instant, new ones take ~1s each)...')
df_street[['lat', 'lng']] = df_street.apply(_geocode_row, axis=1, result_type='expand')
df_geo = df_street.dropna(subset=['lat', 'lng']).copy()
print(f'Geocoded {len(df_geo)} of {len(df_street)} records '
      f'({len(df_street) - len(df_geo)} could not be located)')

In [ ]:
# Build readable hover labels
df_geo['Address'] = (
    df_geo['Property house number'].fillna('').astype(str) + ' ' +
    df_geo['Property street name'].fillna('') + ', ' +
    df_geo['Property locality'].fillna('')
).str.strip()
df_geo['Sale date']       = df_geo['Contract date'].dt.strftime('%d %b %Y')
df_geo['Formatted price'] = df_geo['Purchase price'].apply(
    lambda x: f'${x:,.0f}' if pd.notna(x) else 'N/A'
)
df_geo['Area (m²)'] = df_geo['Area'].apply(
    lambda x: f'{x:,.0f} m²' if pd.notna(x) else 'N/A'
)

fig = px.scatter_mapbox(
    df_geo,
    lat='lat',
    lon='lng',
    color='Purchase price',
    size='Purchase price',
    size_max=30,
    hover_name='Address',
    hover_data={
        'Formatted price': True,
        'Sale date':        True,
        'Area (m²)':        True,
        'Zoning':           True,
        'Purchase price':   False,   # shown via Formatted price instead
        'lat':              False,
        'lng':              False,
    },
    labels={'Formatted price': 'Purchase price', 'Sale date': 'Contract date'},
    color_continuous_scale=px.colors.sequential.Plasma,
    zoom=15,
    mapbox_style='open-street-map',
    title=f'Sales on {query_street}, {query_locality}  |  {query_start_date} → {query_end_date}',
    width=1000,
    height=650,
)
fig.update_layout(margin={'r': 0, 't': 45, 'l': 0, 'b': 0})
fig.show()